# iTelescope premium image set processing, the easier way


**Author: Dave Strickland**

**Version: 0.5.2-alpha1**

This notebook illustrates how to process a premium image set from iTelescope using the AstroPhotography package. The python environment used corresponds to a miniconda emvironment `ap-env.yml`.

The example dataset used is the iTelescope Plan-20 premium dataset of the M101 supernova [SN 2023 ixf](https://en.wikipedia.org/wiki/SN_2023ixf). Note that these files are not provided with this package. However the notebook should work with your own files if you specify a valid fits-file containing directory at the prompt below.

The processing mirrors that of the `itelescope_premium_the_hard_way.piynb` notebook, except using the `ApProcess` class. Although all of the processing stages allow many user-controlled options the default values will be used in this example.

The processing stages consists of:

1. Creating an `ApProcess` instance to perform the processing. Using the same instance for multiple processing stages on the same set of files is recommended, and simplfies identifying which images to process.
2. (To be added later) Optional bad pixel, bad column and bad row removal in the cases where the input images still have such artifacts. This is common in the iTelescope Premium image sets, where their calibration does not remove all artifacts.
3. Finding stars in the image to allow later astrometric solution finding and image quality checking.
4. Astrometric solutions using Astrometry.net
5. Resampling and stacking of multiple images onto a common footprint to create a final image in each available band..
6. Three color image composite creation using images from  mutliple bands.

## Notebook environment setup

In [1]:
import os
import pathlib
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time
import math
import subprocess
import shlex

from ccdproc import ImageFileCollection
from astropy.table import Table
from astropy import wcs
import astropy
from astropy.io import fits

import AstroPhotography as ap

In [2]:
def print_module_version(mod):
    """
    Convenience function to pretty-print Python module mod version.

    Arguments:
        mod {module} -- Imported Python module

    Returns:
        str -- module nam
e and version
    """
    print(f"Using module {mod.__name__:30s}  version: {mod.__version__}")
    return

In [3]:
# Get Version information
print(f'Python version: {sys.version}')
print_module_version(np)
print_module_version(matplotlib)
print_module_version(astropy)
print_module_version(ap)

Python version: 3.10.13 | packaged by conda-forge | (main, Dec 23 2023, 15:36:39) [GCC 12.3.0]
Using module numpy                           version: 1.26.3
Using module matplotlib                      version: 3.8.2
Using module astropy                         version: 6.0.0
Using module AstroPhotography                version: 0.5.2


In [4]:
# Enable inline plotting for graphics
# %matplotlib inline
# Set default figure size to be larger
# this may only work in matplotlib 2.0+!
from IPython.core.interactiveshell import InteractiveShell
matplotlib.rcParams['figure.figsize'] = [10.0, 6.0]
# Enable multiple outputs from jupyter cells
InteractiveShell.ast_node_interactivity = "all"

# Basic Pipeline Processing

## Create an ApProcess instance

For convenience we'll also change into the directory containing the files we intend to process.

In [5]:
default_dir = '/old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline'
print(f'Default directory for input files: {default_dir}')

print('Enter the path to the directory containing the premium image set (or return for the default): ')
wdir = input('Image set path:').strip() or default_dir
try:
    os.chdir(wdir.strip())
    print('Switched directory to ' + os.getcwd())
except:
    print(f'Error, os.chdir threw an exception changing to {wdir}')
    print('Check that the path you supplied is a valid filesystem path.')
    raise

Default directory for input files: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
Enter the path to the directory containing the premium image set (or return for the default): 


Image set path: 


Switched directory to /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline


In [6]:
# Choose the logging level.
loglevel = 'DEBUG'
processor = ap.ApProcess(loglevel)

## Star Detection and Astrometry

At the end of this stage we should have valid astrometric solutions, in the form of valid WCS coordinate headers,
added to copies of the input calibrated images (to avoid modifying the input calibrated images). These files, refrered 
to here as *navigated images*, should work with your favorite FITS viewer to show Right Ascensions and Declinations.

### Basic Parameters

In [7]:
# At the most basic, only data_dir needs to be specified
data_dir        = r'.'

# If the data_dir contains fits files that we do not want processed, then
# you need to specify either the inclusive and/or exclusive file patterns.
#
# Make sure to check that your pattern works on the command line using `ls`
default_include_pattern = "Calibrated-*-?.fits*"
default_exclude_pattern = None

quality_summary_file = 'quality_summary_M101SN.csv'

print(f'Default include file pattern for input files: {default_include_pattern}')
print(f'Default exclude file pattern for input files: {default_exclude_pattern}')

msg = 'Enter new include input file pattern (or return to accept default pattern)'
include_pattern = input(msg).strip() or default_include_pattern
print(f'Using "{include_pattern}" as the input include file pattern.')

msg = 'Enter new exclude input file pattern (or return to accept default pattern)'
exclude_pattern = input(msg).strip() or default_exclude_pattern
print(f'Using "{exclude_pattern}" as the input exclude file pattern.')

Default include file pattern for input files: Calibrated-*-?.fits*
Default exclude file pattern for input files: None


Enter new include input file pattern (or return to accept default pattern) 


Using "Calibrated-*-?.fits*" as the input include file pattern.


Enter new exclude input file pattern (or return to accept default pattern) 


Using "None" as the input exclude file pattern.


### Optional Parameters

In [8]:
# optional parameters
# To be documented at a later time.

## Run star finding

### Input and Output Files

Before we run find_stars for the first time let us check which files it will use as inputs to find_stars. The processing stage is, unsurprisingly, `'inputs'`.

In [9]:
# Show files names that would be generated...
ap_filestate = 'conceptual'

# Note we need to specify that the input_suffix is 'fits.gz' not for the inputs, which
# are picked up by the include_pattern, but for the output file types.

process_stage     = 'input'
no_name_and_dir   = False
with_name_and_dir = True
input_files = processor.get_file_names(process_stage, data_dir, ap_filestate, include_pattern, exclude_pattern, None, None, '.fits.gz', no_name_and_dir)
for file in input_files:
    print(f'  {file}')

2024-09-07 10:18:41,402 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:41,403 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:41,403 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:41,403 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"
2024-09-07 10:18:42,962 | ApProcess | DEBUG | For ap_filetype input the relative directory path is ./
2024-09-07 10:18:42,963 | ApProcess | DEBUG | Note that output files will be a subdirectory ./


  Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-2.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-3.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-2.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-3.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Green-1.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Green-2.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Green-3.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Red-1.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Red-2.fits.gz
  Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Red-3.fits.gz


We can also see what the other output file names will be. **Note** that when processing it is a convention to place the output files in subdirectories. These subdirectories could be the same for a different set of input files, e.g. different input directories representing different observing nights on a specific target but having all the navigated images for that target written to a single output directory.

There are various ways to access the filename and subdirectory information, either separately or together.

You do not need to run these functions before calling  `ApProcess.navigate_images`, they're performed here for purely educational reasons, and can be disabled by setting show_files to False.

In [10]:
# Full set of ftypes
##ftypes = ['srclist', 'regfile', 'plotfile', 'qualfile', 'fwhmplot', 'navfile']

# Shorter subset of the files than must be generated for WCS generated and resampling/stacking to work
ftypes = ['srclist', 'navfile']

show_files = True
if show_files:
    for ftype in ftypes:
        print(80*'=')
        # Get directory info, relative path or absolute path
        rel_filedir = processor.get_directories(ftype, data_dir, False)
        abs_filedir = processor.get_directories(ftype, data_dir, True)
        print(f'File type {ftype} output directory names:')
        print(f'  Relative path={rel_filedir}\n  Absolute path={abs_filedir}')
        print(80*'-')
        
        # File names, without directory path
        print(f'File type {ftype} output file names (no directory info):')
        ofiles = processor.get_file_names(ftype, data_dir, ap_filestate, include_pattern, exclude_pattern, None, None, '.fits.gz', no_name_and_dir)
        print(f'  There are {len(ofiles)} files in the file list.')
        for file in ofiles:
            print(f'  {file}')
        print(80*'-')
            
        # File names **with** directory path
        print(f'File type {ftype} output file names (including directory):')
        ofiles = processor.get_file_names(ftype, data_dir, ap_filestate, include_pattern, exclude_pattern, None, None, '.fits.gz', with_name_and_dir)
        print(f'  There are {len(ofiles)} files in the file list.')
        for file in ofiles:
            print(f'  {file}')

        # Files that this instance has already processed (without directory)
        print(f'File type {ftype} output file names (no directory info), files that have been processed by this instance:')
        ofiles = processor.get_file_names(ftype, data_dir, 'processed', include_pattern, exclude_pattern, None, None, '.fits.gz', no_name_and_dir)
        print(f'  There are {len(ofiles)} files in the file list.')
        for file in ofiles:
            print(f'  {file}')
        print(80*'-')

        # File that exist on disk, even if they haven't been processed by this instance
        print(f'File type {ftype} output file names (no directory info), files that exist on disk:')
        ofiles = processor.get_file_names(ftype, data_dir, 'existing', include_pattern, exclude_pattern, None, None, '.fits.gz', no_name_and_dir)
        print(f'  There are {len(ofiles)} files in the file list.')
        for file in ofiles:
            print(f'  {file}')
        
else:
    print(f'File name display disabled. To enable, set show_files=True')

2024-09-07 10:18:42,968 | ApProcess | DEBUG | For ap_filetype srclist the relative directory path is ./SourceLists/
2024-09-07 10:18:42,969 | ApProcess | DEBUG | For ap_filetype srclist the relative directory path is ./SourceLists/
2024-09-07 10:18:42,969 | ApProcess | DEBUG | For ap_filetype srclist the absolute directory path is /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline/SourceLists
2024-09-07 10:18:42,969 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:42,969 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:42,970 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:42,970 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"


File type srclist output directory names:
  Relative path=./SourceLists/
  Absolute path=/old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline/SourceLists
--------------------------------------------------------------------------------
File type srclist output file names (no directory info):


2024-09-07 10:18:44,464 | ApProcess | DEBUG | For ap_filetype srclist the relative directory path is ./SourceLists/
2024-09-07 10:18:44,465 | ApProcess | DEBUG | Note that output files will be a subdirectory ./SourceLists/
2024-09-07 10:18:44,465 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:44,465 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:44,466 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:44,466 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"


  There are 12 files in the file list.
  srclist-M101_Supernova_2023ixf-180s-Lum-1.fits
  srclist-M101_Supernova_2023ixf-180s-Lum-2.fits
  srclist-M101_Supernova_2023ixf-180s-Lum-3.fits
  srclist-M101_Supernova_2023ixf-300s-Blue-1.fits
  srclist-M101_Supernova_2023ixf-300s-Blue-2.fits
  srclist-M101_Supernova_2023ixf-300s-Blue-3.fits
  srclist-M101_Supernova_2023ixf-300s-Green-1.fits
  srclist-M101_Supernova_2023ixf-300s-Green-2.fits
  srclist-M101_Supernova_2023ixf-300s-Green-3.fits
  srclist-M101_Supernova_2023ixf-300s-Red-1.fits
  srclist-M101_Supernova_2023ixf-300s-Red-2.fits
  srclist-M101_Supernova_2023ixf-300s-Red-3.fits
--------------------------------------------------------------------------------
File type srclist output file names (including directory):


2024-09-07 10:18:45,958 | ApProcess | DEBUG | For ap_filetype srclist the relative directory path is ./SourceLists/
2024-09-07 10:18:45,959 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:45,959 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:45,959 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:45,959 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"


...

  There are 12 files in the file list.
  navigated-M101_Supernova_2023ixf-180s-Lum-1.fits
  navigated-M101_Supernova_2023ixf-180s-Lum-2.fits
  navigated-M101_Supernova_2023ixf-180s-Lum-3.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-1.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-2.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-3.fits
  navigated-M101_Supernova_2023ixf-300s-Green-1.fits
  navigated-M101_Supernova_2023ixf-300s-Green-2.fits
  navigated-M101_Supernova_2023ixf-300s-Green-3.fits
  navigated-M101_Supernova_2023ixf-300s-Red-1.fits
  navigated-M101_Supernova_2023ixf-300s-Red-2.fits
  navigated-M101_Supernova_2023ixf-300s-Red-3.fits
--------------------------------------------------------------------------------
File type navfile output file names (including directory):


2024-09-07 10:18:50,432 | ApProcess | DEBUG | For ap_filetype navfile the relative directory path is ./NavigatedImages/
2024-09-07 10:18:50,432 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:50,432 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:50,433 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:50,433 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"


  There are 12 files in the file list.
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-1.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-2.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-3.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-1.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-2.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-3.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-1.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-2.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-3.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Red-1.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Red-2.fits
  ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Red-3.fits
File type navfile output file names (no directory info), files that have been processed by this instance:
  There are 0 file

2024-09-07 10:18:51,933 | ApProcess | DEBUG | For ap_filetype navfile the relative directory path is ./NavigatedImages/
2024-09-07 10:18:51,934 | ApProcess | DEBUG | Note that output files will be a subdirectory ./NavigatedImages/


  There are 12 files in the file list.
  navigated-M101_Supernova_2023ixf-180s-Lum-1.fits
  navigated-M101_Supernova_2023ixf-180s-Lum-2.fits
  navigated-M101_Supernova_2023ixf-180s-Lum-3.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-1.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-2.fits
  navigated-M101_Supernova_2023ixf-300s-Blue-3.fits
  navigated-M101_Supernova_2023ixf-300s-Green-1.fits
  navigated-M101_Supernova_2023ixf-300s-Green-2.fits
  navigated-M101_Supernova_2023ixf-300s-Green-3.fits
  navigated-M101_Supernova_2023ixf-300s-Red-1.fits
  navigated-M101_Supernova_2023ixf-300s-Red-2.fits
  navigated-M101_Supernova_2023ixf-300s-Red-3.fits


This matches what we expect. The order of the files is unimportant at this stage.

### Navigating the Data

Now run `navigate_images`. There are a large number of optional parameters that we will just accept the recommended default values for.

In [11]:
clean_run = False
if clean_run:
    clean_srclists = True
    clean_navfiles = True
else:
    clean_srclists = False
    clean_navfiles = False

# For initial debugging it it best to make sure exceptions terminate the run.
# And all output is sent to the screen
initial_debugging = True
if initial_debugging:
    stop_on_error = True
    quiet = False
    init_filelist = ['Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz', 
                     'Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz']
    status = processor.navigate_images(data_dir, include_pattern, exclude_pattern, init_filelist, None, '.fits.gz', 
                                   quality_summary_file, clean_srclists, clean_navfiles, stop_on_error=stop_on_error, quiet=quiet)

else:
    # Once the first few files work, the better default is
    stop_on_error = False
    quiet = True

    status = processor.navigate_images(data_dir, include_pattern, exclude_pattern, None, None, '.fits.gz', 
                                   quality_summary_file, clean_srclists, clean_navfiles, stop_on_error=stop_on_error, quiet=quiet)



2024-09-07 10:18:51,937 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:51,938 | ApProcess | INFO | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:51,938 | ApProcess | INFO | Using specified file list: ['Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz', 'Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz']
2024-09-07 10:18:52,191 | ApProcess | INFO | There are 2 input files matching the parameters given.
2024-09-07 10:18:52,191 | ApProcess | DEBUG | Input file collection:
ImageFileCollection(location='.', keywords=['naxis1', 'naxis2', 'imagetyp', 'object', 'filter', 'exposure'], filenames=['Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz', 'Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz'])
2024-09-07 10:18:52,460 | ApProcess | DEBUG | ---------------------------------------------------------------------------

In [12]:
# show status table
status.pprint(max_width=200)

                            filename                             find_stars_status find_stars_time astrometry_status astrometry_time
---------------------------------------------------------------- ----------------- --------------- ----------------- ---------------
 Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz    skipped_exists           0.000    skipped_exists           0.000
Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz    skipped_exists           0.000    skipped_exists           0.000


The processing times for each stage depend on the host processor single CPU speed (single threaded python), but will typically be longer for the astrometry stage as the time depends on the nova.astrometry.net web service. This depends on their CPU oad, but also very strongly on the number of possible locations and scales it needs to consider. `ApAstrometry` tries to provide a number of hints to `Astrometry.net`, the more of which are present the more likely a successful solution in a shorter time.

In particular, if `ApFindStars` can estimate the approximate plate scale of the images, then astrometric solutions are much faster although astrometry is always slower than the source searching. Note that raw and calibrated images from `iTelescope` often have the FITS header keywords needed for approximate plate scale estimation, but they are often stripped out of iTelescope Premium image sets.

In [13]:
# Load and display the quality summary table (NOTE this is a very wide table)
qual_summ_table = Table.read(quality_summary_file, format="ascii.csv")
qual_summ_table.columns   # Dict of table columns (access by column name, index, or slice)
qual_summ_table.colnames  # List of column names
qual_summ_table.meta      # Dict of meta-data
len(qual_summ_table)      # Number of table rows

<TableColumns names=('targ:tel:filter','file','ncols','nrows','filter','median','stddev','num_detected','num_with_photometry','search_nsigma','adups_brightest','adups_median','adups_faintest','num_saturated_in_image','num_saturated_in_photometry','num_fit','circular_psf','fwhm_val_pix','fwhm_err_pix','fwhm_val_arcs','fwhm_err_arcs','num_data_pts')>

['targ:tel:filter',
 'file',
 'ncols',
 'nrows',
 'filter',
 'median',
 'stddev',
 'num_detected',
 'num_with_photometry',
 'search_nsigma',
 'adups_brightest',
 'adups_median',
 'adups_faintest',
 'num_saturated_in_image',
 'num_saturated_in_photometry',
 'num_fit',
 'circular_psf',
 'fwhm_val_pix',
 'fwhm_err_pix',
 'fwhm_val_arcs',
 'fwhm_err_arcs',
 'num_data_pts']

OrderedDict()

12

In [14]:
# Show some data from the quality summary data.
qual_summ_table['file', 'ncols','nrows','filter','median','stddev',
    'num_detected','fwhm_val_pix','fwhm_err_pix','fwhm_val_arcs','fwhm_err_arcs','num_data_pts'].pprint_all()

                               file                               ncols nrows filter   median     stddev  num_detected fwhm_val_pix fwhm_err_pix fwhm_val_arcs fwhm_err_arcs num_data_pts
----------------------------------------------------------------- ----- ----- ------ ---------- --------- ------------ ------------ ------------ ------------- ------------- ------------
 Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-2.fits.gz  3056  3056   Blue 179.874023 21.801167          278     3.465217     0.284413        -999.0        -999.0           50
 Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-3.fits.gz  3056  3056   Blue 183.536133 21.837498          266     3.625976     0.287341        -999.0        -999.0           48
 Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz  3056  3056   Blue 184.711258 21.853487          243     3.883582     0.247384        -999.0        -999.0           48
Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Green-1.fits.gz  305

The quality summary file provides a general overview of the images that have been processed. (Note it summarizes all quality `YaML` files in the specified directory, not just the ones generated in the latest run of the `navigate_images` function.)

- The file is grouped by Filter, even if the lexical order of the files names is different.
   - The sigma-clipped background level, with detected sources removed, `median` +/- `stddev` (standard deviation) is one way to check
     if a particular image is affected by clouds. For example the Blue filter images have very similar background levels of 180+/-22, 184+/-22, and 185+/-22 ADU.
     The third Green filter image, on the other hand, differs from the others in have a background level of 177+/-22 compared to 233+/-23 and 223+/-23.
   - The number of detected sources should likewise be similar, and again the third Green image differs from the first and second Green images.
- The most important statistic are the measured source full width at half maximum, a measure of the telescope resolution, seeing conditions and tracking stability. Outliers here should like be excluded from image stacking.
   - In this case only the values in pixel units, not arcseconds are valid. That is because the pixel-to-arcsecond conversion used by ApMeasureStars comes
     from plate scale FITS header keywords. The are often missing from  iTelescope Premium datasets.

#### Looking at the WCS headers

We can look at the WCS headers of the navigated files we just generated to assess what the WCS solutions are like, e.g.

In [15]:
# List of navigated file processed by this instance
navigated_file_list = processor.get_file_names('navfile', data_dir, 'processed', include_pattern, exclude_pattern, None, None, '.fits.gz', with_name_and_dir)
print(f'Navigated image files generated by this ApProcess instance:  \n{navigated_file_list}')

navigated_file_list = processor.get_file_names('navfile', data_dir, 'existing', include_pattern, exclude_pattern, None, None, '.fits.gz', with_name_and_dir)
print(f'Navigated image files on disk:  \n{navigated_file_list}')

first_file = navigated_file_list[0]
w, pix_scales, pix_area, im_scales = ap.util.load_wcs_from_file(first_file)
print(f'\nSummarizing WCS information from {first_file}:\n')
ap.util.summarize_wcs(w)

2024-09-07 10:18:52,852 | ApProcess | DEBUG | For ap_filetype navfile the relative directory path is ./NavigatedImages/
2024-09-07 10:18:52,853 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:52,853 | ApProcess | DEBUG | Attempting to find stars in FITS files within directory=.
2024-09-07 10:18:52,853 | ApProcess | DEBUG | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:52,853 | ApProcess | DEBUG | Excluding files that match exclude_pattern="None"


Navigated image files generated by this ApProcess instance:  
['navigated-M101_Supernova_2023ixf-180s-Lum-1.fits', 'navigated-M101_Supernova_2023ixf-300s-Blue-1.fits']
Navigated image files on disk:  
['./NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-1.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-2.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-180s-Lum-3.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-1.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-2.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-3.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-1.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-2.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Green-3.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Red-1.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Red-2.fits', './NavigatedImages/

### Checking the Navigated Images

It is recommended that you check that the WCS headers give reasonable results and are self consistent. I personally prefer to use `SAOImage ds9` for this from the command line, by loading the images, aligning them and then blinking between the images. If the astrometry is good then the same star should be in the same place on the screen in each image, and not move or jump from frame to frame.

```bash
> cd ./NavigatedImages/
> ls
navigated-M101_Supernova_2023ixf-180s-Lum-1.fits   navigated-M101_Supernova_2023ixf-300s-Green-1.fits
navigated-M101_Supernova_2023ixf-180s-Lum-2.fits   navigated-M101_Supernova_2023ixf-300s-Green-2.fits
navigated-M101_Supernova_2023ixf-180s-Lum-3.fits   navigated-M101_Supernova_2023ixf-300s-Green-3.fits
navigated-M101_Supernova_2023ixf-300s-Blue-1.fits  navigated-M101_Supernova_2023ixf-300s-Red-1.fits
navigated-M101_Supernova_2023ixf-300s-Blue-2.fits  navigated-M101_Supernova_2023ixf-300s-Red-2.fits
navigated-M101_Supernova_2023ixf-300s-Blue-3.fits  navigated-M101_Supernova_2023ixf-300s-Red-3.fits
> ds9  -asinh -zoom to fit -cmap viridis -scale mode 99.5 navigated-M101_Supernova_2023ixf-*fits &
```

Then:
- expand the ds9 window to make it larger
- click on the first image (or any image) so that it is outlined in blue
- Mousing over pixels in that image should change the WCS RA and Dec values shown
- Click the menu `Zoom` -> `Align` and the first image and the zoomed-out view in the top right should change orientation
- Cleck `Frame` -> `Single Frame`, then `Zoom` -> `Fit`
- Now to match all the images by RA and Dec, `Frame` -> `Match` -> `Frame` -> `WCS`
- Make sure the default blink interval is 1 second (less than 1 second is too fast) by `Frame` -> `Frame Parameters` -> `Blink Interval`
- Finally, set the images to blink using `Frame` ->  `Blink Frames`


## Image Stacking (Resampling and Mosaicing)

`ApProcessor` can also resample images to a common WCS system, and mosaic multiple images sharing a common WCS together, using `Astromatic` `swarp` under the hood. These operations are more commonly referred to as image stacking in the amateur astronomy community.

As discussed in the `itelescope_premium_the_hard_way.ipynb` notebook there are a number of ways of doing this, but for now we only support the first method listed there: Resample all images to match the WCS of another image, for example the first image in the list of navigated images, irrespective of how uniform the platescale is or whether the image is North-up, East-left aligned.

The `ApProcess.resample_images_to_match` function provides this method. By default it processes all the navigated files in all available `FILTER` combinations that the current `ApProcess` instance is aware of, irrespective of whether the files have the same exposure time, which may not be ideal in cases where you have images with different seeing quality and or different exposure times.

If cases where you already have pre-existing navigated images not processesed by your current `ApProcess` instance you will need to either use `ApProcess.set_file_names` or the 

In [16]:
unique_filters = processor.get_filter_list()
print(unique_filters)

2024-09-07 10:18:54,352 | ApProcess | DEBUG | get_filter_list: there are 2 files of type input found under .
2024-09-07 10:18:54,354 | ApProcess | DEBUG | Finding files and exposure times for filter Blue
2024-09-07 10:18:54,604 | ApProcess | DEBUG | File Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz uses filter Blue
2024-09-07 10:18:54,605 | ApProcess | INFO | Found the following 1 files for filter Blue: ['Calibrated-iTelescope-M101_Supernova_2023ixf-300s-Blue-1.fits.gz']
2024-09-07 10:18:54,605 | ApProcess | DEBUG | Finding files and exposure times for filter Lum
2024-09-07 10:18:54,729 | ApProcess | DEBUG | File Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz uses filter Lum
2024-09-07 10:18:54,852 | ApProcess | INFO | Found the following 1 files for filter Lum: ['Calibrated-iTelescope-M101_Supernova_2023ixf-180s-Lum-1.fits.gz']
2024-09-07 10:18:54,853 | ApProcess | DEBUG | Unique filter and exposure time data:
2024-09-07 10:18:54,853 | ApProcess | 

['Blue', 'Lum']


*Note* that the ImageFileCollection shows only two files, because we set `initial_debugging=True` and only processed two images.

To process any other images that already exist we can use `ApProcess.set_file_names`.

In [17]:
processor.set_file_names(data_dir, include_pattern, exclude_pattern, None, None, '.fits.gz')

2024-09-07 10:18:54,856 | ApProcess | INFO | Finding input and output files starting in directory=.
2024-09-07 10:18:54,857 | ApProcess | DEBUG | Current working directory: /old_lnx/home/dks/Downloads/iTelescopeScratch/Plan-20-M101SN-Pipeline
2024-09-07 10:18:54,857 | ApProcess | INFO | Using files that match include_pattern="Calibrated-*-?.fits*"
2024-09-07 10:18:54,857 | ApProcess | INFO | Excluding files that match exclude_pattern="None"
2024-09-07 10:18:56,352 | ApProcess | INFO | There are 12 input files matching the parameters given.
2024-09-07 10:18:56,352 | ApProcess | DEBUG | Input file collection:
ImageFileCollection(location='.', keywords=['naxis1', 'naxis2', 'imagetyp', 'object', 'filter', 'exposure'], glob_include='Calibrated-*-?.fits*')
2024-09-07 10:18:56,352 | ApProcess | INFO | Looking for existing files matching name conventions...
2024-09-07 10:18:56,353 | ApProcess | DEBUG | Looking for existing srclist files.
2024-09-07 10:18:56,353 | ApProcess | INFO | Found the f

In [18]:
unique_filters = processor.get_filter_list('navfile')
print(f'unique_filters = {unique_filters}')

2024-09-07 10:18:56,372 | ApProcess | DEBUG | get_filter_list: there are 12 files of type navfile found under .
2024-09-07 10:18:56,374 | ApProcess | DEBUG | Finding files and exposure times for filter Blue
2024-09-07 10:18:56,376 | ApProcess | DEBUG | File ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-1.fits uses filter Blue
2024-09-07 10:18:56,376 | ApProcess | DEBUG | File ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-2.fits uses filter Blue
2024-09-07 10:18:56,377 | ApProcess | DEBUG | File ./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-3.fits uses filter Blue
2024-09-07 10:18:56,379 | ApProcess | INFO | Found the following 3 files for filter Blue: ['./NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-1.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-2.fits', './NavigatedImages/navigated-M101_Supernova_2023ixf-300s-Blue-3.fits']
2024-09-07 10:18:56,379 | ApProcess | DEBUG | Finding files and exposure times fo

unique_filters = ['Blue', 'Green', 'Lum', 'Red']


Having dispensed with an example of how to find which filters are available, let's just let `resample_images_to_match` process all the images.

In [19]:
# Todo

# Versions and Changes

| Version | Date | Description |
|:--------|------|-------------|
| 0.5.2-alpha1 | 2024-04-26 | Created script skeleton. |